# 🤖 QLoRA Fine-Tuning: Qwen2.5-7B → Resume Parser

**Task:** Fine-tune Qwen2.5-7B-Instruct (4-bit QLoRA) để chuyển đổi text CV thô (OCR) → JSON template có cấu trúc.

**Pipeline (10 bước):**
1. `PREPROCESS DATA` – Format ChatML + tokenize (với chat_template của Qwen)
2. `MAIN PIPELINE` – Config inline, set seed, kiểm tra GPU
3. `LOAD MODEL + TOKENIZER` – 4-bit BitsAndBytes + LoRA target modules
4. `LOAD DATASET` – Đọc ShareGPT JSON từ Kaggle input
5. `TOKENIZE DATA` – Apply Qwen chat_template lên toàn bộ dataset
6. `DATA COLLATOR` – DataCollatorForLanguageModeling (Causal LM)
7. `METRICS` – Perplexity + ROUGE-L trên validation set
8. `TRAINING CONFIG` – SFTConfig với QLoRA hyperparams
9. `TRAINER & MONITOR` – SFTTrainer + ResourceMonitorCallback → CSV
10. `SAVE MODEL` – Lưu LoRA adapters + merge full model

---
**GPU khuyến nghị:** Kaggle P100 (16 GB) hoặc T4 x2 (2×16 GB)

**Dataset input:** `/kaggle/input/cv-resume-dataset/train_dataset_sharegpt.json`

---
## ⚙️ STEP 0 – Cài đặt thư viện (chỉ chạy 1 lần khi khởi động notebook)

In [ ]:
import subprocess, sys

pkgs = [
    # Core fine-tuning stack
    "transformers>=4.45.0",
    "trl>=0.12.0",             # SFTTrainer
    "peft>=0.13.0",            # LoRA / QLoRA
    "bitsandbytes>=0.43.0",    # 4-bit quantization
    "accelerate>=0.34.0",
    "datasets>=2.20.0",
    # Metrics
    "evaluate>=0.4.2",
    "rouge_score>=0.1.2",
    # Tokenizer dependency cho Qwen
    "tiktoken",
    # Resource monitor
    "psutil",
    "pynvml",
]

print("📦 Đang cài đặt thư viện...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs
)
print("✅ Cài đặt hoàn tất!")

---
## 📦 STEP 1 – PREPROCESS DATA

Định nghĩa hàm `build_preprocess_fn` để:
- Áp dụng **Qwen2.5 ChatML template** lên từng cặp (human, gpt)
- Tokenize cả đoạn hội thoại với `apply_chat_template`
- Mask phần **instruction (human turn)** trong labels → loss chỉ tính trên phần trả lời JSON
- Tự động **truncate** nếu vượt `MAX_SEQ_LENGTH` để tránh OOM

In [ ]:
import torch
import numpy as np

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1 : PREPROCESS DATA
# ═══════════════════════════════════════════════════════════════════════════════

def build_preprocess_fn(tokenizer, max_seq_length: int, train_on_responses_only: bool = True):
    """
    Xây dựng hàm tiền xử lý cho từng conversation (human → gpt).

    Logic:
      1. Dùng tokenizer.apply_chat_template() để format toàn bộ conversation
         theo đúng ChatML format của Qwen2.5:
             <|im_start|>system\n...\n<|im_end|>
             <|im_start|>user\n{cv_text}<|im_end|>
             <|im_start|>assistant\n{json_output}<|im_end|>
      2. Tokenize với truncation để tránh OOM.
      3. Nếu train_on_responses_only=True:
         Mask labels = -100 cho toàn bộ phần instruction (system + user turn)
         → Loss chỉ được tính trên phần JSON output (assistant turn).
         Điều này giúp model học sinh JSON thay vì học lặp lại instruction.

    Args:
        tokenizer:               Qwen2.5 tokenizer đã load.
        max_seq_length:          Số token tối đa (truncate nếu vượt).
        train_on_responses_only: Nếu True, mask instruction khỏi loss.

    Returns:
        preprocess_fn (callable): Hàm nhận 1 sample dict → dict đã tokenize.
    """

    SYSTEM_PROMPT = (
        "Bạn là một chuyên gia phân tích hồ sơ nhân sự (Resume Parser). "
        "Hãy đọc đoạn text CV (OCR) được cung cấp và trích xuất thông tin "
        "thành đúng định dạng JSON theo schema mẫu."
    )

    # Token ID dùng để nhận diện ranh giới assistant turn
    # Qwen2.5 dùng: <|im_start|>assistant\n
    ASSISTANT_HEADER_IDS = tokenizer.encode(
        "<|im_start|>assistant\n", add_special_tokens=False
    )
    RESPONSE_START_LEN = len(ASSISTANT_HEADER_IDS)

    def find_response_start(input_ids: list) -> int:
        """Tìm vị trí bắt đầu của assistant turn trong danh sách token IDs."""
        for i in range(len(input_ids) - RESPONSE_START_LEN + 1):
            if input_ids[i : i + RESPONSE_START_LEN] == ASSISTANT_HEADER_IDS:
                return i + RESPONSE_START_LEN  # Trả về vị trí SAU header
        return len(input_ids)  # Nếu không tìm thấy, mask toàn bộ

    def preprocess_fn(sample: dict) -> dict:
        conversations = sample.get("conversations", [])

        # ── Xây dựng messages theo ChatML ────────────────────────────────────
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        for turn in conversations:
            role = "user" if turn["from"] == "human" else "assistant"
            messages.append({"role": role, "content": turn["value"]})

        # ── Apply Qwen chat template → full text string ───────────────────────
        # add_generation_prompt=False vì đã có assistant turn đầy đủ trong data
        full_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        # ── Tokenize ──────────────────────────────────────────────────────────
        tokenized = tokenizer(
            full_text,
            max_length=max_seq_length,
            truncation=True,   # Cắt ngắn để tránh OOM
            padding=False,     # DataCollator sẽ lo padding
            return_tensors=None,
        )

        input_ids = tokenized["input_ids"]

        # ── Mask labels (chỉ train trên phần assistant response) ──────────────
        if train_on_responses_only:
            response_start = find_response_start(input_ids)
            labels = [-100] * response_start + input_ids[response_start:]
        else:
            labels = input_ids.copy()

        tokenized["labels"] = labels
        return tokenized

    return preprocess_fn


print("✅ [Step 1] Hàm build_preprocess_fn đã sẵn sàng.")

---
## 🔧 STEP 2 – MAIN PIPELINE: Config & Setup

- Khai báo tất cả **hyperparameters** inline (thay cho YAML file trên Kaggle)
- Cố định **seed** để kết quả reproducible
- Kiểm tra GPU, in thông tin VRAM

In [ ]:
import os, sys, csv, json, time, random, threading, warnings
import numpy as np
import torch
from transformers import set_seed

warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2 : MAIN PIPELINE – CONFIG & SETUP
# ═══════════════════════════════════════════════════════════════════════════════

# ── Đường dẫn Kaggle ─────────────────────────────────────────────────────────
INPUT_DIR      = "/kaggle/input/cv-resume-dataset"         # Thay bằng tên dataset Kaggle của bạn
OUTPUT_DIR     = "/kaggle/working/resume_parser_qlora"     # Thư mục lưu kết quả
DATASET_FILE   = os.path.join(INPUT_DIR, "train_dataset_sharegpt.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL_ID  = "Qwen/Qwen2.5-7B-Instruct"  # Model gốc trên HuggingFace Hub

# ── Token length ──────────────────────────────────────────────────────────────
# CV text thường dài ~800-1500 token sau khi format ChatML
# JSON output ~200-600 token → Tổng cộng đặt 2048
MAX_SEQ_LENGTH = 2048

# ── Dataset split ─────────────────────────────────────────────────────────────
DEV_RATIO      = 0.1    # 10% dữ liệu dùng cho validation
SEED           = 42

# ── LoRA hyperparameters ──────────────────────────────────────────────────────
LORA_R         = 16     # Rank của LoRA matrix (càng cao càng mạnh nhưng tốn VRAM)
LORA_ALPHA     = 32     # Scaling factor: thường = 2 * lora_r
LORA_DROPOUT   = 0.05
# Target modules của Qwen2.5 (attention + MLP projection layers)
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",   # Attention
    "gate_proj", "up_proj", "down_proj",        # MLP / FFN
]

# ── Training hyperparameters ──────────────────────────────────────────────────
NUM_TRAIN_EPOCHS              = 3
PER_DEVICE_TRAIN_BATCH_SIZE   = 2     # Kaggle P100: 2-4; T4: 1-2
PER_DEVICE_EVAL_BATCH_SIZE    = 4
GRADIENT_ACCUMULATION_STEPS   = 8     # Effective batch = 2 * 8 = 16
LEARNING_RATE                 = 2e-4  # Phổ biến cho QLoRA
WEIGHT_DECAY                  = 0.01
LR_SCHEDULER_TYPE             = "cosine"
WARMUP_RATIO                  = 0.05
MAX_GRAD_NORM                 = 1.0

# ── Precision ─────────────────────────────────────────────────────────────────
# Kaggle P100 hỗ trợ fp16; A100/T4 hỗ trợ bf16
USE_FP16       = True
USE_BF16       = False

# ── Eval & Checkpoint ─────────────────────────────────────────────────────────
EVAL_STRATEGY             = "epoch"
SAVE_STRATEGY             = "epoch"
LOGGING_STEPS             = 20
SAVE_TOTAL_LIMIT          = 2
LOAD_BEST_MODEL_AT_END    = True
METRIC_FOR_BEST_MODEL     = "eval_loss"
EARLY_STOPPING_PATIENCE   = 2
TRAIN_ON_RESPONSES_ONLY   = True   # Chỉ tính loss trên phần JSON output

# ── Resource Monitor ──────────────────────────────────────────────────────────
MONITOR_INTERVAL_SECONDS  = 30

# ── Debug flags ───────────────────────────────────────────────────────────────
SMOKE_TEST     = False   # True = chỉ lấy 8 mẫu để test pipeline
OVERFIT_TEST   = False   # True = train + eval trên cùng 16 mẫu

# ── Cố định seed ──────────────────────────────────────────────────────────────
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Kiểm tra GPU ──────────────────────────────────────────────────────────────
print("=" * 60)
print("  Resume Parser QLoRA Fine-Tuning – Qwen2.5-7B")
print("=" * 60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🟢 GPU     : {gpu_name}")
    print(f"🟢 VRAM    : {vram_gb:.1f} GB")
else:
    print("🔴 CẢNH BÁO: Không tìm thấy GPU! QLoRA cần GPU để chạy.")
    print("   → Vào Settings → Accelerator → chọn GPU P100 hoặc T4")

print(f"📁 Dataset : {DATASET_FILE}")
print(f"📁 Output  : {OUTPUT_DIR}")
print(f"🔒 Seed    : {SEED}")
print(f"📏 MaxLen  : {MAX_SEQ_LENGTH} tokens")
print(f"🔁 Epochs  : {NUM_TRAIN_EPOCHS}")
print(f"⚡ LoRA r={LORA_R}, α={LORA_ALPHA}")
print("=" * 60)

---
## 🧠 STEP 3 – LOAD MODEL + TOKENIZER

- Load **Qwen2.5-7B-Instruct** với `BitsAndBytesConfig` (4-bit NF4 quantization)
- Bọc model với **LoRA adapters** qua `get_peft_model()`
- In số tham số trainable (thường ~0.5-1% so với full model)

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3 : LOAD MODEL + TOKENIZER
# ═══════════════════════════════════════════════════════════════════════════════

print("[Step 3] Cấu hình 4-bit quantization (BitsAndBytes NF4)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                        # Bật 4-bit quantization
    bnb_4bit_quant_type="nf4",                # NF4 (Normal Float 4) – tốt hơn fp4
    bnb_4bit_compute_dtype=torch.bfloat16,    # Compute trong bfloat16 để tăng tốc
    bnb_4bit_use_double_quant=True,           # Double quantization giảm thêm VRAM
)

print(f"[Step 3] Tải tokenizer từ: {BASE_MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
    padding_side="right",   # Qwen2.5 cần padding_side='right' cho SFT
)
# Qwen2.5 đã có eos_token, nhưng cần đảm bảo pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"[Step 3] Tải model Qwen2.5-7B-Instruct (4-bit)... (có thể mất 2-5 phút)")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",           # Tự động phân bổ layers lên GPU
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager", # "flash_attention_2" nếu Kaggle GPU hỗ trợ
)

# Chuẩn bị model cho k-bit training (bật gradient checkpointing + cast layer norms)
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,  # Tiết kiệm VRAM bằng cách recompute activations
)

# ── Cấu hình LoRA ─────────────────────────────────────────────────────────────
print("[Step 3] Gắn LoRA adapters...")
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",              # Không train bias để giảm tham số
    inference_mode=False,
)

model = get_peft_model(model, lora_config)

# ── In thống kê tham số ───────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
pct = 100 * trainable_params / total_params

print(f"[Step 3] ✅ Model đã sẵn sàng!")
print(f"         Tổng tham số      : {total_params:,}")
print(f"         Tham số trainable  : {trainable_params:,} ({pct:.3f}%)")
print(f"         VRAM đang dùng    : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
model.print_trainable_parameters()

---
## 📂 STEP 4 – LOAD DATASET

- Đọc file `train_dataset_sharegpt.json` từ Kaggle input
- Tự động tách train/dev theo tỉ lệ `DEV_RATIO`
- Hỗ trợ **smoke_test** (8 mẫu) và **overfit_test** (16 mẫu)

In [ ]:
from datasets import load_dataset, DatasetDict, Dataset

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4 : LOAD DATASET
# ═══════════════════════════════════════════════════════════════════════════════

print(f"[Step 4] Đọc dữ liệu từ: {DATASET_FILE}")

if not os.path.exists(DATASET_FILE):
    raise FileNotFoundError(
        f"Không tìm thấy file dataset tại: {DATASET_FILE}\n"
        "→ Hãy upload file train_dataset_sharegpt.json lên Kaggle Datasets "
        "và thêm vào notebook qua: Add data → Your datasets."
    )

raw_dataset = load_dataset("json", data_files=DATASET_FILE, split="train")
print(f"[Step 4] Đã đọc: {len(raw_dataset):,} mẫu")

# ── Debug modes ───────────────────────────────────────────────────────────────
if SMOKE_TEST:
    print("[Step 4] ⚠  SMOKE TEST – chỉ lấy 8 mẫu để kiểm tra pipeline nhanh.")
    raw_dataset = raw_dataset.select(range(8))

elif OVERFIT_TEST:
    print("[Step 4] ⚠  OVERFIT TEST – train và eval trên cùng 16 mẫu.")
    raw_dataset = raw_dataset.select(range(min(16, len(raw_dataset))))

# ── Tách train/dev ────────────────────────────────────────────────────────────
if OVERFIT_TEST:
    split_dataset = DatasetDict({
        "train":      raw_dataset,
        "validation": raw_dataset,
    })
else:
    split = raw_dataset.train_test_split(
        test_size=DEV_RATIO,
        seed=SEED,
        shuffle=True,
    )
    split_dataset = DatasetDict({
        "train":      split["train"],
        "validation": split["test"],
    })

print(f"[Step 4] ✅ Train: {len(split_dataset['train']):,} mẫu | "
      f"Dev: {len(split_dataset['validation']):,} mẫu")

# Preview 1 mẫu
sample = split_dataset["train"][0]
convs  = sample.get("conversations", [])
if convs:
    print(f"\n📋 Preview mẫu đầu tiên:")
    print(f"   Human (300 chars): {convs[0]['value'][:300]}...")
    print(f"   GPT   (200 chars): {convs[1]['value'][:200]}...")

---
## 🔤 STEP 5 – TOKENIZE DATA

- Dùng `.map()` để áp `preprocess_fn` lên toàn bộ dataset
- Xóa cột `conversations` sau khi tokenize (giải phóng RAM)
- Filter bỏ các mẫu quá ngắn (< 10 token)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5 : TOKENIZE DATA
# ═══════════════════════════════════════════════════════════════════════════════

print("[Step 5] Áp dụng tokenize lên toàn bộ dataset...")

preprocess_fn = build_preprocess_fn(
    tokenizer=tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
    train_on_responses_only=TRAIN_ON_RESPONSES_ONLY,
)

# Các cột văn bản gốc cần xóa sau khi tokenize
cols_to_remove = split_dataset["train"].column_names

tokenized_datasets = split_dataset.map(
    preprocess_fn,
    batched=False,             # Xử lý từng mẫu một (an toàn hơn với chat_template)
    remove_columns=cols_to_remove,
    desc="Tokenizing",
    num_proc=1,                # Không dùng multiprocessing trên Kaggle để tránh lỗi
)

# Lọc bỏ mẫu quá ngắn (lỗi format) hoặc toàn -100 trong labels
def is_valid_sample(example):
    input_ids = example["input_ids"]
    labels    = example["labels"]
    # Phải có ít nhất 10 token và ít nhất 1 label không phải -100
    has_content = len(input_ids) >= 10
    has_label   = any(l != -100 for l in labels)
    return has_content and has_label

before_train = len(tokenized_datasets["train"])
before_dev   = len(tokenized_datasets["validation"])

tokenized_datasets = tokenized_datasets.filter(
    is_valid_sample, desc="Filtering invalid samples"
)

after_train = len(tokenized_datasets["train"])
after_dev   = len(tokenized_datasets["validation"])

print(f"[Step 5] ✅ Tokenize hoàn tất!")
print(f"         Train  : {before_train:,} → {after_train:,} (bỏ {before_train - after_train} mẫu)")
print(f"         Dev    : {before_dev:,}   → {after_dev:,}   (bỏ {before_dev - after_dev} mẫu)")

# Thống kê độ dài token
sample_len = tokenized_datasets["train"][0]["input_ids"]
all_lengths = [len(tokenized_datasets["train"][i]["input_ids"])
               for i in range(min(100, after_train))]
print(f"         Độ dài trung bình (100 mẫu đầu): {np.mean(all_lengths):.0f} tokens")
print(f"         Độ dài max: {max(all_lengths)} | min: {min(all_lengths)}")

---
## 🗂️ STEP 6 – DATA COLLATOR

Dùng `DataCollatorForLanguageModeling` để:
- Tự động **padding** các sequence trong cùng batch về độ dài bằng nhau
- Đặt `mlm=False` vì đây là Causal LM (không phải Masked LM như BERT)

In [ ]:
from transformers import DataCollatorForLanguageModeling

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6 : DATA COLLATOR
# ═══════════════════════════════════════════════════════════════════════════════

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,           # Causal LM (next-token prediction), không phải Masked LM
    pad_to_multiple_of=8,  # Tối ưu tensor cores trên GPU NVIDIA
)

print("[Step 6] ✅ DataCollatorForLanguageModeling (mlm=False) đã sẵn sàng.")
print("         → Padding tự động | pad_to_multiple_of=8 (tensor core optimization)")

---
## 📊 STEP 7 – METRICS

Định nghĩa `compute_metrics` với:
- **Perplexity**: Đo độ không chắc chắn của model (exp của cross-entropy loss). Càng thấp càng tốt.
- **ROUGE-L**: Đo độ khớp chuỗi dài nhất giữa JSON prediction và JSON ground truth.

> 💡 Với Causal LM, metric quan trọng nhất vẫn là `eval_loss` (cross-entropy). ROUGE được tính bằng cách sinh (generate) ra JSON và so sánh.


In [ ]:
import evaluate
import math

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7 : METRICS
# ═══════════════════════════════════════════════════════════════════════════════

rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
    """
    Tính metrics trên validation set.

    Input eval_preds:
      - predictions: logits hoặc generated token IDs (shape: [B, seq_len, vocab])
      - labels:      ground truth token IDs (shape: [B, seq_len])
                     với -100 tại các vị trí padding/masked

    Metrics trả về:
      - perplexity: exp(mean cross-entropy loss) – thước đo "độ bối rối" của model
      - rouge1, rouge2, rougeL: Đo độ khớp JSON output (n-gram và chuỗi chung)
      - gen_len: Độ dài trung bình của phần prediction
    """
    predictions, labels = eval_preds

    # Nếu predictions là logits (3D) → argmax để lấy token IDs
    if predictions.ndim == 3:
        pred_ids = np.argmax(predictions, axis=-1)
    else:
        pred_ids = predictions

    # ── Decode predictions ────────────────────────────────────────────────────
    # Thay -100 bằng pad_token_id trước khi decode
    pred_ids_clean = np.where(pred_ids != -100, pred_ids, tokenizer.pad_token_id)
    decoded_preds  = tokenizer.batch_decode(pred_ids_clean, skip_special_tokens=True)

    # ── Decode labels ─────────────────────────────────────────────────────────
    labels_clean  = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels_clean, skip_special_tokens=True)

    # Dọn khoảng trắng dư
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    # ── Tính Perplexity từ loss ───────────────────────────────────────────────
    # (Trainer sẽ tự log eval_loss; ở đây tính perplexity từ đó)
    # Perplexity = exp(loss) → chỉ hợp lệ khi predictions là logits
    perplexity = None
    if predictions.ndim == 3:
        try:
            import torch.nn.functional as F
            logits_t = torch.tensor(predictions, dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.long)
            # Chỉ tính loss tại các vị trí label != -100
            loss = F.cross_entropy(
                logits_t.view(-1, logits_t.size(-1)),
                labels_t.view(-1),
                ignore_index=-100,
                reduction="mean",
            )
            perplexity = round(math.exp(loss.item()), 4)
        except Exception:
            perplexity = None

    # ── Tính ROUGE ────────────────────────────────────────────────────────────
    rouge_result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=False,
    )

    # Độ dài trung bình prediction (không tính padding)
    pred_lens = [np.count_nonzero(p != tokenizer.pad_token_id) for p in pred_ids_clean]

    result = {
        "rouge1"  : round(rouge_result.get("rouge1", 0.0) * 100, 4),
        "rouge2"  : round(rouge_result.get("rouge2", 0.0) * 100, 4),
        "rougeL"  : round(rouge_result.get("rougeL", 0.0) * 100, 4),
        "gen_len" : round(np.mean(pred_lens), 2),
    }
    if perplexity is not None:
        result["perplexity"] = perplexity

    return result


print("[Step 7] ✅ Metrics: Perplexity + ROUGE-1/2/L đã sẵn sàng.")

---
## ⚙️ STEP 8 – TRAINING CONFIG

Khai báo `SFTConfig` (kế thừa `TrainingArguments`) với đầy đủ hyperparameters:
- `gradient_checkpointing=True` để tiết kiệm VRAM
- `optim="paged_adamw_8bit"` – optimizer 8-bit cho QLoRA
- `max_seq_length` để SFTTrainer tự động xử lý sequence packing

In [ ]:
from trl import SFTConfig

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 8 : TRAINING CONFIG
# ═══════════════════════════════════════════════════════════════════════════════

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,

    # ── Training loop ────────────────────────────────────────────────────────
    num_train_epochs            = NUM_TRAIN_EPOCHS,

    # ── Batch size ───────────────────────────────────────────────────────────
    per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size  = PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
    # Effective batch size = per_device * gradient_accum * num_gpus

    # ── Optimizer ────────────────────────────────────────────────────────────
    learning_rate               = LEARNING_RATE,
    weight_decay                = WEIGHT_DECAY,
    lr_scheduler_type           = LR_SCHEDULER_TYPE,
    warmup_ratio                = WARMUP_RATIO,
    max_grad_norm               = MAX_GRAD_NORM,
    # paged_adamw_8bit: Adam optimizer 8-bit được tối ưu cho QLoRA
    # Tiết kiệm ~50% VRAM so với Adam thường
    optim                       = "paged_adamw_8bit",

    # ── Mixed precision ──────────────────────────────────────────────────────
    fp16                        = USE_FP16,
    bf16                        = USE_BF16,

    # ── Memory optimization ──────────────────────────────────────────────────
    gradient_checkpointing      = True,  # Recompute activations thay vì lưu → tiết kiệm VRAM
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    dataloader_num_workers      = 0,     # 0 trên Kaggle để tránh lỗi multiprocessing

    # ── SFT specific ─────────────────────────────────────────────────────────
    max_seq_length              = MAX_SEQ_LENGTH,
    dataset_text_field          = None,   # Dataset đã pre-tokenized
    packing                     = False,  # Không pack nhiều conversations vào 1 sequence
    dataset_kwargs              = {"skip_prepare_dataset": True},  # Dataset đã tokenized sẵn

    # ── Evaluation & Checkpoint ──────────────────────────────────────────────
    evaluation_strategy         = EVAL_STRATEGY,
    save_strategy               = SAVE_STRATEGY,
    logging_strategy            = "steps",
    logging_steps               = LOGGING_STEPS,
    save_total_limit            = SAVE_TOTAL_LIMIT,
    load_best_model_at_end      = LOAD_BEST_MODEL_AT_END,
    metric_for_best_model       = METRIC_FOR_BEST_MODEL,
    greater_is_better           = False,  # eval_loss: càng thấp càng tốt

    # ── Reproducibility ──────────────────────────────────────────────────────
    seed                        = SEED,
    data_seed                   = SEED,

    # ── Logging ──────────────────────────────────────────────────────────────
    report_to                   = "none",  # Không cần WandB trên Kaggle
    logging_dir                 = os.path.join(OUTPUT_DIR, "logs"),

    # ── Misc ─────────────────────────────────────────────────────────────────
    remove_unused_columns       = False,
    label_names                 = ["labels"],
)

print("[Step 8] ✅ SFTConfig đã sẵn sàng.")
eff_batch = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
print(f"         Effective batch size : {eff_batch}")
print(f"         Optimizer            : paged_adamw_8bit")
print(f"         Learning rate        : {LEARNING_RATE}")
print(f"         Scheduler            : {LR_SCHEDULER_TYPE} (warmup {WARMUP_RATIO*100:.0f}%)")
print(f"         fp16={USE_FP16} | bf16={USE_BF16} | grad_ckpt=True")

---
## 🚀 STEP 9 – TRAINER & MONITOR

- Định nghĩa **ResourceMonitorCallback**: chạy trên background thread, ghi CPU/RAM/VRAM/nhiệt độ GPU ra `training_monitor.csv` mỗi 30 giây
- Khởi tạo **SFTTrainer** với đầy đủ model, config, dataset, collator, metrics
- Gọi **`trainer.train()`** để bắt đầu training

In [ ]:
import psutil
from transformers import TrainerCallback, TrainerControl, TrainerState, EarlyStoppingCallback

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 9a : RESOURCE MONITOR CALLBACK
# ═══════════════════════════════════════════════════════════════════════════════

class ResourceMonitorCallback(TrainerCallback):
    """
    Callback theo dõi tài nguyên hệ thống trong quá trình training.

    Chạy trên background thread (daemon), ghi vào CSV sau mỗi N giây:
      • step          – Bước huấn luyện hiện tại
      • timestamp     – Thời điểm ghi
      • cpu_percent   – % CPU usage
      • ram_used_gb   – RAM đang dùng (GB)
      • ram_percent   – % RAM usage
      • gpu_vram_used_mb  – VRAM đang dùng (MB)
      • gpu_vram_total_mb – Tổng VRAM (MB)
      • gpu_temp_c    – Nhiệt độ GPU (°C, nếu pynvml khả dụng)
    """

    def __init__(self, output_dir: str, interval_seconds: int = 30):
        self.output_dir    = output_dir
        self.interval      = interval_seconds
        self._stop_event   = threading.Event()
        self._thread       = None
        self._step         = 0
        self._csv_path     = os.path.join(output_dir, "training_monitor.csv")
        self._header_done  = False

        # Thử khởi tạo pynvml để đọc nhiệt độ GPU
        self._nvml = None
        self._nvml_handle = None
        try:
            import pynvml
            pynvml.nvmlInit()
            self._nvml        = pynvml
            self._nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
            print(f"[Monitor] ✅ pynvml khởi tạo OK – sẽ ghi nhiệt độ GPU.")
        except Exception as e:
            print(f"[Monitor] ⚠  pynvml không khả dụng ({e}) – bỏ qua nhiệt độ GPU.")

    def _collect(self) -> dict:
        stat = {
            "timestamp"        : time.strftime("%Y-%m-%d %H:%M:%S"),
            "step"             : self._step,
            "cpu_percent"      : psutil.cpu_percent(interval=None),
            "ram_used_gb"      : round(psutil.virtual_memory().used / 1e9, 2),
            "ram_percent"      : psutil.virtual_memory().percent,
            "gpu_vram_used_mb" : 0,
            "gpu_vram_total_mb": 0,
            "gpu_temp_c"       : 0,
        }
        if torch.cuda.is_available():
            stat["gpu_vram_used_mb"]  = round(torch.cuda.memory_allocated() / 1e6, 1)
            stat["gpu_vram_total_mb"] = round(
                torch.cuda.get_device_properties(0).total_memory / 1e6, 1
            )
        if self._nvml and self._nvml_handle:
            try:
                stat["gpu_temp_c"] = self._nvml.nvmlDeviceGetTemperature(
                    self._nvml_handle, self._nvml.NVML_TEMPERATURE_GPU
                )
            except Exception:
                pass
        return stat

    def _write(self, stat: dict):
        os.makedirs(self.output_dir, exist_ok=True)
        is_new = not os.path.exists(self._csv_path) or not self._header_done
        with open(self._csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(stat.keys()))
            if is_new:
                writer.writeheader()
                self._header_done = True
            writer.writerow(stat)

    def _loop(self):
        while not self._stop_event.is_set():
            self._write(self._collect())
            self._stop_event.wait(timeout=self.interval)

    def on_train_begin(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        print(f"[Monitor] Bắt đầu ghi tài nguyên vào: {self._csv_path}")
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def on_step_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        self._step = state.global_step

    def on_train_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        self._stop_event.set()
        if self._thread:
            self._thread.join(timeout=5)
        print(f"[Monitor] Đã dừng. File CSV: {self._csv_path}")


print("[Step 9a] ✅ ResourceMonitorCallback đã sẵn sàng.")

In [ ]:
from trl import SFTTrainer

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 9b : KHỞI TẠO TRAINER
# ═══════════════════════════════════════════════════════════════════════════════

print("[Step 9b] Khởi tạo SFTTrainer...")

callbacks = [ResourceMonitorCallback(
    output_dir=OUTPUT_DIR,
    interval_seconds=MONITOR_INTERVAL_SECONDS
)]

if EARLY_STOPPING_PATIENCE > 0:
    callbacks.append(EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE))
    print(f"[Step 9b] EarlyStopping patience={EARLY_STOPPING_PATIENCE} đã bật.")

trainer = SFTTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized_datasets["train"],
    eval_dataset    = tokenized_datasets["validation"],
    tokenizer       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = callbacks,
)

print("[Step 9b] ✅ SFTTrainer đã sẵn sàng.")
print(f"          Model      : Qwen2.5-7B-Instruct (4-bit QLoRA)")
print(f"          Train set  : {len(tokenized_datasets['train']):,} mẫu")
print(f"          Val set    : {len(tokenized_datasets['validation']):,} mẫu")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 9c : BẮT ĐẦU HUẤN LUYỆN
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("  ▶  Bắt đầu QLoRA fine-tuning Qwen2.5-7B...")
print("=" * 60)

# Xóa cache GPU trước khi train
torch.cuda.empty_cache()

train_result = trainer.train()

# ── In tóm tắt kết quả ────────────────────────────────────────────────────────
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

print("\n" + "=" * 60)
print("  Training Summary")
print("=" * 60)
for k, v in metrics.items():
    print(f"  {k:40s}: {v}")

if "train_loss" in metrics:
    try:
        ppl = math.exp(metrics["train_loss"])
        print(f"  {'train_perplexity':40s}: {ppl:.4f}")
    except Exception:
        pass

---
## 💾 STEP 10 – SAVE MODEL

Hai lựa chọn lưu model:

1. **Lưu LoRA adapters** (nhẹ, ~100-500 MB) → để tiếp tục fine-tune hoặc inference với PEFT
2. **Merge + Save full model** (nặng, ~14 GB) → để export sang GGUF hoặc deploy

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 10 : SAVE MODEL
# ═══════════════════════════════════════════════════════════════════════════════

# ── 10a: Lưu LoRA Adapters (khuyến nghị cho Kaggle) ──────────────────────────
lora_output_dir = os.path.join(OUTPUT_DIR, "lora_adapters")
print(f"[Step 10a] Lưu LoRA adapters → {lora_output_dir}")

trainer.model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)

print(f"[Step 10a] ✅ LoRA adapters đã lưu! (file nhỏ, chỉ ~{LORA_R * 2 * 7 / 1000:.0f} MB)")

# ── 10b: Merge LoRA weights vào base model (tuỳ chọn) ────────────────────────
MERGE_AND_SAVE = True   # Đặt False nếu không đủ VRAM/RAM để merge

if MERGE_AND_SAVE:
    print("\n[Step 10b] Merge LoRA weights vào base model (cần thêm VRAM)...")
    merged_output_dir = os.path.join(OUTPUT_DIR, "merged_model")

    try:
        from peft import PeftModel

        # Tải lại base model với full precision để merge
        print("  → Tải lại base model...")
        base_model_for_merge = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )

        # Load LoRA adapters lên base model
        peft_model = PeftModel.from_pretrained(base_model_for_merge, lora_output_dir)

        # Merge và unload LoRA weights
        print("  → Merging...")
        merged_model = peft_model.merge_and_unload()

        # Lưu full merged model
        os.makedirs(merged_output_dir, exist_ok=True)
        merged_model.save_pretrained(merged_output_dir, safe_serialization=True)
        tokenizer.save_pretrained(merged_output_dir)

        del base_model_for_merge, peft_model, merged_model
        torch.cuda.empty_cache()

        print(f"[Step 10b] ✅ Merged model đã lưu → {merged_output_dir}")

    except Exception as e:
        print(f"[Step 10b] ⚠  Không thể merge ({e})")
        print("           → Bạn vẫn có LoRA adapters tại:", lora_output_dir)

# ── Tóm tắt cuối ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  🎉 Pipeline hoàn thành!")
print("=" * 60)
print(f"  LoRA adapters   : {lora_output_dir}")
if MERGE_AND_SAVE:
    print(f"  Merged model    : {merged_output_dir}")
print(f"  Monitor CSV     : {os.path.join(OUTPUT_DIR, 'training_monitor.csv')}")
print(f"  Training metrics: {os.path.join(OUTPUT_DIR, 'train_results.json')}")
print("\n  → Tải file LoRA adapters về máy để inference với script 02_test_local_model.py")
print("=" * 60)

---
## 🔍 BONUS – Quick Inference Test

Chạy thử model vừa train với 1 đoạn text CV mẫu để kiểm tra chất lượng JSON output.

In [ ]:
# ── Quick Inference sau khi train ─────────────────────────────────────────────

SAMPLE_CV = """
John Smith
john.smith@email.com | +1-555-123-4567 | linkedin.com/in/johnsmith

SUMMARY
Senior Software Engineer with 8+ years of experience in full-stack development.
Expertise in Python, JavaScript, and cloud technologies.

EXPERIENCE
Senior Software Engineer | Google | 2020 - Present
- Led development of distributed systems serving 10M+ users
- Mentored junior engineers and conducted technical interviews

Software Engineer | Amazon | 2017 - 2020
- Developed microservices using Python and AWS Lambda
- Reduced API latency by 40%

EDUCATION
B.S. Computer Science, MIT, 2017

SKILLS
Python, JavaScript, TypeScript, AWS, Docker, Kubernetes, PostgreSQL
"""

SYSTEM_PROMPT = (
    "Bạn là một chuyên gia phân tích hồ sơ nhân sự (Resume Parser). "
    "Hãy đọc đoạn text CV (OCR) được cung cấp và trích xuất thông tin "
    "thành đúng định dạng JSON theo schema mẫu."
)

messages = [
    {"role": "system",    "content": SYSTEM_PROMPT},
    {"role": "user",      "content": f"TEXT CV:\n{SAMPLE_CV}"},
]

text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

print("🔍 Đang inference...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print("\n" + "=" * 60)
print("  📋 JSON Output của Model:")
print("=" * 60)
print(generated)

# Thử parse JSON để kiểm tra tính hợp lệ
try:
    parsed = json.loads(generated)
    print("\n✅ JSON hợp lệ! Các trường:", list(parsed.keys()))
except json.JSONDecodeError as e:
    print(f"\n⚠  JSON không hợp lệ: {e}")